<a href="https://colab.research.google.com/github/enkhbayarmunkhbayar1016-creator/Cudo-and-thread-and-omp/blob/main/biydaaltcuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install & Setup
!apt-get update -qq
!apt-get install -y -qq build-essential nvidia-cuda-toolkit

cuda_code = """
#include <iostream>
#include <vector>
#include <random>
#include <cstdio>
#include <cuda_runtime.h>
using namespace std;

__global__ void oddEven(int* a, int n, int phase){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int idx = phase + 2 * i;
    if(idx + 1 < n && a[idx] > a[idx+1]){
        int t = a[idx];
        a[idx] = a[idx+1];
        a[idx+1] = t;
    }
}

int main(){
    int sizes[] = {10000, 100000, 1000000};
    int sz = 3;

    remove("results_cuda.json");

    cudaDeviceProp p;
    cudaGetDeviceProperties(&p, 0);
    fprintf(stderr, "GPU: %s\\n", p.name);

    for(int i = 0; i < sz; i++){
        int n = sizes[i];
        vector<int> a(n);

        mt19937 g(42);
        uniform_int_distribution<> d(0, 100000);
        for(int j = 0; j < n; j++) a[j] = d(g);

        fprintf(stderr, "CUDA n=%d...\\n", n); fflush(stderr);

        int *dev;
        cudaMalloc(&dev, n * sizeof(int));
        cudaMemcpy(dev, a.data(), n * sizeof(int), cudaMemcpyHostToDevice);

        cudaEvent_t t0, t1;
        cudaEventCreate(&t0);
        cudaEventCreate(&t1);

        int blk = 256;
        int grid = (n/2 + blk - 1) / blk;

        cudaEventRecord(t0);
        for(int pp = 0; pp < n; pp++){
            oddEven<<<grid, blk>>>(dev, n, pp % 2);
        }
        cudaEventRecord(t1);
        cudaEventSynchronize(t1);

        float ms = 0;
        cudaEventElapsedTime(&ms, t0, t1);

        cudaMemcpy(a.data(), dev, n * sizeof(int), cudaMemcpyDeviceToHost);
        cudaFree(dev);
        cudaEventDestroy(t0);
        cudaEventDestroy(t1);

        bool ok = true;
        for(int j = 0; j + 1 < n; j++) if(a[j] > a[j+1]) {ok = false; break;}

        fprintf(stderr, "  %.2f ms\\n", ms);

        char q = '"';
        FILE* f = fopen("results_cuda.json", "a");
        fprintf(f, "{%cn%c:%d,%ctime_ms%c:%.4f,%csorted%c:%s,%cdevice%c:%cGPU%c}\\n",
                q, q, n, q, q, (double)ms, q, q, ok?"true":"false", q, q, q, q);
        fclose(f);
    }
    fprintf(stderr, "CUDA done!\\n");
    return 0;
}
"""

with open("cuda_sort.cu", "w") as f:
    f.write(cuda_code)

print("✓ CUDA code written")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✓ CUDA code written


In [ ]:
# Cell 2 — Compile & Run
import subprocess
import time

print("🔨 Compiling CUDA...")
r = subprocess.run(["nvcc", "-O2", "-o", "cuda", "cuda_sort.cu"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(f" Compile failed:\n{r.stderr}")
else:
    print("✓ Compiled OK")

print("\n▶ Running benchmark (10k, 100k, 1M)...")
t0 = time.time()
r = subprocess.run(["./cuda"], capture_output=True, text=True)
print(r.stderr)
print(f"⏱ Total: {time.time()-t0:.1f}s")

🔨 Compiling CUDA...
✓ Compiled OK

▶ Running benchmark (10k, 100k, 1M)...
GPU: Tesla T4
CUDA n=10000...
  115.99 ms
CUDA n=100000...
  324.55 ms
CUDA n=1000000...
  12859.21 ms
CUDA done!

⏱ Total: 13.7s


In [ ]:
# Cell 3 — Save to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Copy results to Drive
shutil.copy("results_cuda.json", "/content/drive/My Drive/results_cuda.json")

print("✓ Saved to Google Drive: /My Drive/results_cuda.json")

Mounted at /content/drive
✓ Saved to Google Drive: /My Drive/results_cuda.json


In [ ]:
# Cell 4 — Download
from google.colab import files
files.download("results_cuda.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>